In [10]:
import requests, json

API_KEY = "7cb1b1-daf77f-94ed27"
NAME = "Ezra G Goldstein"

url = "https://app.overton.io/documents.php"
params = {"query": f"\"{NAME}\"", "format": "json", "api_key": API_KEY}

titles = []
while True:
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    for d in data.get("results", []):
        for a in d.get("authors", []) or []:
            # Check if a is a dictionary (detailed author info) or string (organization name)
            if isinstance(a, dict):
                if a.get("type") == "person" and a.get("name") == NAME:
                    titles.append(d.get("title"))
                    break
            elif isinstance(a, str) and a == NAME:
                titles.append(d.get("title"))
                break

    next_url = (data.get("query") or {}).get("next_page_url")
    if not next_url:
        break
    # next_page_url is a full URL; reset for next request
    url = next_url.split("?", 1)[0]
    params = dict(x.split("=", 1) for x in next_url.split("?", 1)[1].split("&"))

print("\n".join(titles))


# Overton API Examples

The Overton API allows you to search for policy documents and their citations. Here are some key things you can do:

1. **Search by author name** (like above)
2. **Search by DOI** - find policy documents that cite specific research papers
3. **Search by keywords** - find policy documents about specific topics
4. **Search by organization** - find documents from specific institutions
5. **Get citation counts** - see how often research is cited in policy

Let's try a few different examples:

In [7]:
# Example 1: Search for policy documents mentioning "climate change"
import requests

API_KEY = "7cb1b1-daf77f-94ed27"

url = "https://app.overton.io/documents.php"
params = {
    "query": "climate change",
    "format": "json", 
    "api_key": API_KEY,
    "limit": 5  # Just get first 5 results
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
data = r.json()

print(f"Found {data.get('query', {}).get('total', 0)} total documents about climate change")
print("\nFirst 5 document titles:")
for i, doc in enumerate(data.get("results", [])[:5], 1):
    print(f"{i}. {doc.get('title', 'No title')}")
    print(f"   Source: {doc.get('source', {}).get('name', 'Unknown source')}")
    print(f"   Year: {doc.get('published_date', 'Unknown')[:4] if doc.get('published_date') else 'Unknown'}")
    print()

Found 0 total documents about climate change

First 5 document titles:
1. Global Climate Change Alliance
   Source: Unknown source
   Year: Unknown

2. Climate Change Adaptation: U.S. Department of Agriculture
   Source: Unknown source
   Year: Unknown

3. Practical consideration of climate change
   Source: Unknown source
   Year: Unknown

4. Under the Weather and the Rising Tide: Adapting to a Changing Climate in Asia and the Pacific
   Source: Unknown source
   Year: Unknown

5. Naturally Resilient—MNRF’s Natural Resource Climate Adaptation Strategy.
   Source: Unknown source
   Year: Unknown



In [12]:
# Example 2: Find policy documents that cite a specific research paper
# Let's use a real DOI from a climate science paper

API_KEY = "7cb1b1-daf77f-94ed27"
DOI = "10.1038/nature14240"  # A famous climate change paper

url = "https://app.overton.io/documents.php"
params = {
    "plain_dois_cited": DOI,
    "format": "json", 
    "api_key": API_KEY,
    "limit": 5
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
data = r.json()

print(f"Found {data.get('query', {}).get('total', 0)} policy documents that cite DOI: {DOI}")
print("\nPolicy documents citing this research:")
for i, doc in enumerate(data.get("results", [])[:5], 1):
    print(f"{i}. {doc.get('title', 'No title')}")
    print(f"   Source: {doc.get('source', {}).get('name', 'Unknown source')}")
    if doc.get('policy_areas'):
        areas = [area.get('name', '') for area in doc.get('policy_areas', [])]
        print(f"   Policy areas: {', '.join(areas[:3])}{'...' if len(areas) > 3 else ''}")
    print()

Found 0 policy documents that cite DOI: 10.1038/nature14240

Policy documents citing this research:
1. The Economic and Health Consequences of Climate Change
   Source: Unknown source



In [9]:
# Example 3: Search for documents from a specific organization
API_KEY = "7cb1b1-daf77f-94ed27"

url = "https://app.overton.io/documents.php"
params = {
    "source": "WHO",  # World Health Organization
    "format": "json", 
    "api_key": API_KEY,
    "limit": 5
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
data = r.json()

print(f"Found {data.get('query', {}).get('total', 0)} documents from WHO")
print("\nRecent WHO policy documents:")
for i, doc in enumerate(data.get("results", [])[:5], 1):
    print(f"{i}. {doc.get('title', 'No title')}")
    if doc.get('abstract'):
        # Show first 100 characters of abstract
        abstract = doc.get('abstract', '')[:100] + "..." if len(doc.get('abstract', '')) > 100 else doc.get('abstract', '')
        print(f"   Abstract: {abstract}")
    print()

Found 0 documents from WHO

Recent WHO policy documents:


## Key Overton API Capabilities

Based on the examples above, here's what you can do with the Overton API:

### Main Endpoints:
1. **`/documents.php`** - Search for policy documents
2. **`/generate_id_set.php`** - Create sets of DOIs for batch queries

### Search Parameters:
- **`query`** - Free text search (keywords, phrases, author names)
- **`plain_dois_cited`** - Find documents citing specific DOIs
- **`source`** - Filter by organization/source
- **`policy_areas`** - Filter by policy domain
- **`published_date_from/to`** - Date range filtering
- **`limit`** - Number of results per page

### What You Can Find:
- **Policy documents** that cite academic research
- **Citation relationships** between research and policy
- **Policy areas** where research has impact
- **Organizations** producing policy documents
- **Authors** of policy documents
- **Geographic coverage** of policies

### Use Cases:
1. **Research Impact Assessment** - See how your research influences policy
2. **Literature Reviews** - Find policy context for research topics  
3. **Stakeholder Mapping** - Identify organizations working on specific issues
4. **Policy Monitoring** - Track policy developments in your field